# Figure 6 and hierarchy-system functional annotation

This notebook annotates genes retained by the hierarchy-scoring system with reviewed
UniProt records.

## Workflow

1. load hierarchy scores;
2. retain genes with positive scores;
3. map Ensembl gene identifiers to Ensembl transcript identifiers using GENCODE;
4. match transcript identifiers against reviewed human UniProt records;
5. export one representative annotation per scored gene.

The original notebook contained no complete Figure 6 plotting workflow. This cleaned
version therefore focuses on the functional-annotation step that was actually present
in the source file.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


## Configuration

Update `project_dir` when running the notebook in a different environment.
The optional gene-list file can be used to restrict annotation to a predefined subset.


In [ ]:
project_dir = Path("/data1/HOMO_PANGENOME/DYY/data")
bambu_dir = project_dir / "bambu"

score_file = bambu_dir / "s14.csv"
gencode_gtf_file = bambu_dir / "ref" / "gencode.v42.annotation.gtf"
uniprot_file = (
    project_dir
    / "uniprot_database"
    / "uniprotkb_Human_AND_reviewed_true_AND_m.tsv"
)
gene_name_file = project_dir / "ensg_gene_name.csv"
optional_gene_list_file = bambu_dir / "123.txt"

output_dir = bambu_dir / "figure_6_hierarchy_system"
output_dir.mkdir(parents=True, exist_ok=True)

minimum_score = 0


## Utility functions


In [ ]:
def require_file(file_path):
    """Raise an informative error when an input file is missing."""
    if not file_path.is_file():
        raise FileNotFoundError(f"Required file was not found: {file_path}")


def require_columns(table, required_columns, table_name):
    """Validate that a table contains all required columns."""
    missing_columns = sorted(set(required_columns) - set(table.columns))

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing required columns: {missing_columns}"
        )


def remove_ensembl_version(identifier):
    """Remove a terminal Ensembl version suffix such as '.12'."""
    if pd.isna(identifier):
        return np.nan

    return re.sub(r"\.\d+$", "", str(identifier).strip())


def parse_gtf_attributes(attribute_text):
    """Parse the semicolon-delimited attribute field of a GTF record."""
    attributes = {}

    for item in str(attribute_text).strip().strip(";").split(";"):
        item = item.strip()

        if not item or " " not in item:
            continue

        key, value = item.split(" ", 1)
        attributes[key] = value.strip().strip('"')

    return attributes


## Load hierarchy scores


In [ ]:
require_file(score_file)

score_table = pd.read_csv(score_file, sep="\t")
require_columns(
    score_table,
    {"ensg", "score"},
    "Hierarchy-score table",
)

score_table = score_table.copy()
score_table["ensg"] = score_table["ensg"].map(
    remove_ensembl_version
)
score_table["score"] = pd.to_numeric(
    score_table["score"],
    errors="coerce",
)

score_table = (
    score_table.dropna(subset=["ensg", "score"])
    .loc[lambda table: table["score"] > minimum_score]
    .sort_values("score", ascending=False)
    .drop_duplicates("ensg")
)

if score_table.empty:
    raise ValueError(
        "No genes remained after applying the hierarchy-score threshold."
    )


## Optionally restrict the scored genes

When `123.txt` exists, only gene identifiers listed in that file are retained.
This replaces the original notebook behavior in which `ensg_list` was silently
overwritten midway through the analysis.


In [ ]:
if optional_gene_list_file.is_file():
    selected_genes = pd.read_csv(
        optional_gene_list_file,
        header=None,
        names=["ensg"],
    )

    selected_genes["ensg"] = selected_genes["ensg"].map(
        remove_ensembl_version
    )

    score_table = score_table.loc[
        score_table["ensg"].isin(selected_genes["ensg"])
    ].copy()

    if score_table.empty:
        raise ValueError(
            "The optional gene list did not overlap the positive-score genes."
        )


## Parse GENCODE transcript annotations


In [ ]:
require_file(gencode_gtf_file)

gtf_columns = [
    "seqname",
    "source",
    "feature",
    "start",
    "end",
    "score",
    "strand",
    "frame",
    "attributes",
]

gtf_table = pd.read_csv(
    gencode_gtf_file,
    sep="\t",
    comment="#",
    names=gtf_columns,
    dtype={"seqname": str},
)

transcript_gtf = gtf_table.loc[
    gtf_table["feature"].eq("transcript"),
    ["attributes"],
].copy()

parsed_attributes = transcript_gtf["attributes"].map(
    parse_gtf_attributes
)

gene_transcript_map = pd.DataFrame(
    {
        "ensg": parsed_attributes.map(
            lambda item: item.get("gene_id")
        ),
        "enst": parsed_attributes.map(
            lambda item: item.get("transcript_id")
        ),
        "gtf_gene_name": parsed_attributes.map(
            lambda item: item.get("gene_name")
        ),
        "transcript_type": parsed_attributes.map(
            lambda item: (
                item.get("transcript_type")
                or item.get("transcript_biotype")
            )
        ),
    }
)

gene_transcript_map["ensg"] = gene_transcript_map["ensg"].map(
    remove_ensembl_version
)
gene_transcript_map["enst"] = gene_transcript_map["enst"].map(
    remove_ensembl_version
)

gene_transcript_map = gene_transcript_map.dropna(
    subset=["ensg", "enst"]
).drop_duplicates()


## Load reviewed UniProt records

The UniProt `Ensembl` field may contain multiple transcript identifiers. It is expanded
to one transcript–protein match per row before joining with GENCODE.


In [ ]:
require_file(uniprot_file)

uniprot_table = pd.read_csv(
    uniprot_file,
    sep="\t",
    low_memory=False,
)

require_columns(
    uniprot_table,
    {"Ensembl"},
    "UniProt table",
)

uniprot_table = uniprot_table.loc[
    uniprot_table["Ensembl"].notna()
].copy()


def extract_ensembl_transcripts(value):
    """Extract all ENST identifiers from a UniProt Ensembl field."""
    return sorted(
        {
            remove_ensembl_version(identifier)
            for identifier in re.findall(
                r"ENST\d+(?:\.\d+)?",
                str(value),
            )
        }
    )


uniprot_table["enst"] = uniprot_table["Ensembl"].map(
    extract_ensembl_transcripts
)

uniprot_transcript_table = (
    uniprot_table.explode("enst")
    .dropna(subset=["enst"])
    .drop_duplicates()
)


## Add gene symbols when available


In [ ]:
gene_name_table = pd.DataFrame(
    columns=["ensg", "gene_name"]
)

if gene_name_file.is_file():
    raw_gene_name_table = pd.read_csv(
        gene_name_file,
        sep="\t",
        header=None,
    )

    if raw_gene_name_table.shape[1] < 2:
        raise ValueError(
            "The gene-name file must contain at least two columns."
        )

    gene_name_table = raw_gene_name_table.iloc[:, :2].copy()
    gene_name_table.columns = ["ensg", "gene_name"]
    gene_name_table["ensg"] = gene_name_table["ensg"].map(
        remove_ensembl_version
    )
    gene_name_table = gene_name_table.drop_duplicates("ensg")


## Match scored genes to reviewed UniProt records

The join is performed through explicit Ensembl gene and transcript identifiers rather
than repeated substring searches.


In [ ]:
scored_transcripts = score_table.merge(
    gene_transcript_map,
    on="ensg",
    how="left",
    validate="one_to_many",
)

annotated_records = scored_transcripts.merge(
    uniprot_transcript_table,
    on="enst",
    how="left",
    suffixes=("", "_uniprot"),
)

if not gene_name_table.empty:
    annotated_records = annotated_records.merge(
        gene_name_table,
        on="ensg",
        how="left",
    )
else:
    annotated_records["gene_name"] = np.nan

annotated_records["gene_name"] = (
    annotated_records["gene_name"]
    .fillna(annotated_records["gtf_gene_name"])
)


## Select one representative annotation per gene

When several reviewed UniProt records match the same gene, records with a non-missing
UniProt accession are prioritized. The full transcript-level table is also retained.


In [ ]:
accession_candidates = [
    "Entry",
    "Entry Name",
    "From",
]

accession_column = next(
    (
        column
        for column in accession_candidates
        if column in annotated_records.columns
    ),
    None,
)

if accession_column is not None:
    annotated_records["has_uniprot_match"] = (
        annotated_records[accession_column].notna()
    )
else:
    annotated_records["has_uniprot_match"] = (
        annotated_records["Ensembl"].notna()
    )

annotated_records = annotated_records.sort_values(
    ["score", "has_uniprot_match", "ensg", "enst"],
    ascending=[False, False, True, True],
)

representative_annotations = (
    annotated_records
    .drop_duplicates("ensg")
    .copy()
)

representative_annotations["annotation_status"] = np.where(
    representative_annotations["has_uniprot_match"],
    "Reviewed UniProt match",
    "No reviewed UniProt match",
)


## Export reproducible result tables


In [ ]:
annotated_records.to_csv(
    output_dir / "hierarchy_gene_uniprot_transcript_annotations.tsv",
    sep="\t",
    index=False,
)

representative_annotations.to_csv(
    output_dir / "hierarchy_gene_uniprot_representative_annotations.tsv",
    sep="\t",
    index=False,
)

annotation_summary = (
    representative_annotations["annotation_status"]
    .value_counts()
    .rename_axis("annotation_status")
    .reset_index(name="gene_count")
)

annotation_summary.to_csv(
    output_dir / "hierarchy_annotation_summary.tsv",
    sep="\t",
    index=False,
)

annotation_summary


## Reproducibility notes

- All notebook outputs and execution counters were cleared.
- The undefined `enst_df` initialization in the original notebook was removed.
- Bare `except` statements and repeated substring searches were replaced with explicit
  identifier parsing and table joins.
- Ensembl version suffixes are normalized before matching.
- The hierarchy-score table is no longer silently overwritten by a second gene list.
- Both complete transcript-level matches and one representative record per gene are
  exported.
